In [2]:
import os
import time
from getpass import getpass
from google import genai
from google.genai.errors import ServerError


class Project:
    def __init__(self):
        self.__database = {}  # stores {email: [name, password]}
        api_key = os.getenv("GEMINI_API_KEY") or getpass("Enter your Gemini API key: ")
        self.__client = genai.Client(api_key=api_key)
        self.__firstmenu()

    def __firstmenu(self):
        print("\n=== Welcome to Govinda NLP Services ===")
        first_input = input("""
Hii how would you like to proceed?
1. Not a member? Register 
2. Already a member? Login 
3. Exit 
Enter your choice: """).strip()

        menu_map = {"1": "1. Register", "2": "2. Login", "3": "3. Exit"}
        print(f"Selected Option: {menu_map.get(first_input, first_input)}")

        if first_input == "1":
            self.__register()
        elif first_input == "2":
            self.__login()
        else:
            print("Exiting Govinda NLP Services. Goodbye!")
            exit()

    def __register(self):
        print("\n--- Registration ---")
        name = input("Enter your name : ")
        print(f"Name Entered: {name}")

        email = input("Enter your email address : ")
        print(f"Email Entered: {email}")

        password = input("Enter the password : ")
        print(f"Password Entered: {'*' * len(password)}")

        if email in self.__database:
            print("Data already exists")
        else:
            self.__database[email] = [name, password]
            print("Account successfully registered")
        self.__firstmenu()

    def __login(self):
        print("\n--- Login ---")
        email = input("Enter the email : ")
        print(f"Email Entered: {email}")

        password = input("Enter the password : ")
        print(f"Password Entered: {'*' * len(password)}")

        if email not in self.__database:
            print("Email does not exist, please register")
            self.__firstmenu()
        else:
            if self.__database[email][1] == password:
                print(f"Welcome {self.__database[email][0]}!")
                self.__secondmenu()
            else:
                print("Wrong password, try again")
                self.__login()

    def __secondmenu(self):
        second_input = input("""
How would you like to proceed?
1. NER (Named Entity Recognition)
2. Language Detection
3. Sentiment Analysis
4. Logout
Enter your choice: """).strip()

        menu_map = {
            "1": "1. NER (Named Entity Recognition)",
            "2": "2. Language Detection",
            "3": "3. Sentiment Analysis",
            "4": "4. Logout"
        }
        print(f"Selected Option: {menu_map.get(second_input, second_input)}")

        if second_input == "1":
            self.__ner()
        elif second_input == "2":
            self.__language_detection()
        elif second_input == "3":
            self.__sentiment_analysis()
        else:
            print("Logging out...")
            self.__firstmenu()

    def __ask_gemini(self, prompt):
            for attempt in range(3):
                try:
                    response = self.__client.models.generate_content(
                        model="gemini-3.8-flash",
                        contents=prompt
                    )
                    return response.text.strip()
                except ServerError:
                    time.sleep(2 ** attempt)  # waits 1s, then 2s, then 4s
            return None  # signals caller to use offline fallback

    def __ner(self):
        print("\n--- Named Entity Recognition (NER) ---")
        para = input("Enter the paragraph : ")
        print(f"Paragraph Entered: {para}")

        search_term = input("What would you like to search : ")
        print(f"Search Entity Entered: {search_term}")

        prompt = (
            f"Perform Named Entity Recognition on the paragraph below and extract all "
            f"entities matching '{search_term}'. Return only a clean list of found entities "
            f"(or 'No matching entities found'):\n\n{para}"
        )
        result = self.__ask_gemini(prompt)
        print(f"\nNER Result:\n{result}\n" + "-" * 40)
        self.__secondmenu()

    def __language_detection(self):
        print("\n--- Language Detection ---")
        para = input("Enter the paragraph : ")
        print(f"Paragraph Entered: {para}")

        prompt = (
            "Detect the natural language(s) used in the following paragraph "
            "(such as English, Hindi, French, Spanish, Hinglish, etc.). "
            "Return the detected language name and confidence percentage:\n\n"
            f"{para}"
        )
        result = self.__ask_gemini(prompt)
        print(f"\nLanguage Detection Result:\n{result}\n" + "-" * 40)
        self.__secondmenu()

    def __sentiment_analysis(self):
        print("\n--- Sentiment Analysis ---")
        para = input("Enter the paragraph : ")
        print(f"Paragraph Entered: {para}")

        prompt = (
            "Analyze the sentiment of the following paragraph. "
            "Return the overall sentiment label (Positive, Negative, or Neutral), "
            "confidence percentage, and a brief 1-line explanation:\n\n"
            f"{para}"
        )
        result = self.__ask_gemini(prompt)
        print(f"\nSentiment Analysis Result:\n{result}\n" + "-" * 40)
        self.__secondmenu()

In [ ]:
obj = Project()


=== Welcome to Govinda NLP Services ===
Selected Option: 1. Register

--- Registration ---
Name Entered: Govinda
Email Entered: abc@gmail.com
Password Entered: ****
Account successfully registered

=== Welcome to Govinda NLP Services ===
Selected Option: 2. Login

--- Login ---
Email Entered: abc@gmail.com
Password Entered: ****
Welcome Govinda!
Selected Option: 3. Sentiment Analysis

--- Sentiment Analysis ---
Paragraph Entered: Today the day was great untill it rain and spoiled it 

Sentiment Analysis Result:
**Sentiment:** Negative  
**Confidence:** 85%  
**Explanation:** The initial positive tone is overridden by disappointment when the rain ruins the day.
----------------------------------------
Selected Option: 1. NER (Named Entity Recognition)

--- Named Entity Recognition (NER) ---
Paragraph Entered: Sundar Pichai leads Google in Mountain View, while Satya Nadella runs Microsoft in Redmond and Elon Musk oversees Tesla and SpaceX.
Search Entity Entered: companies

NER Result:
*

: 